# Refresh Opportunity Scoring for Search Content


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jacksonochieng1540/flyrank-ml-capstone/blob/main/notebooks/capstone_refresh_scoring.ipynb?flush_cache=true)

## Capstone Project

This project investigates whether search performance signals can identify content that is likely declining and may require a content refresh.

The goal is to support content review decisions by ranking pages according to their likelihood of decline.

This project uses the FlyRank ML Internship dataset and follows the Refresh / Content Opportunity Scoring lane.

In [21]:
import duckdb
import pandas as pd
import numpy as np
import os

from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

# Set the Hugging Face token as an environment variable
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# The previous attempt to set hf_token via con.execute() was causing an error.
# The httpfs extension should now pick up the token from the environment variable.

## Read the warehouse datacell

In [19]:
daily = con.sql("""
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

daily.head()

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 401)

## build a feature table

In [ ]:
data = con.sql("""
WITH feature_table AS (

SELECT

client_hash_id,

content_hash_id,

SUM(gsc_impressions) AS impressions,

SUM(gsc_clicks) AS clicks,

AVG(gsc_avg_position) AS avg_position,

COUNT(DISTINCT query_hash_id) AS visible_queries,

STDDEV(gsc_avg_position) AS position_volatility

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)

GROUP BY
client_hash_id,
content_hash_id

)

SELECT *
FROM feature_table
""").df()

data.head()

## create the label

In [ ]:
data["ctr"] = (
    data["clicks"] /
    data["impressions"].replace(0, np.nan)
)

data["ctr"] = data["ctr"].fillna(0)

data["low_ctr"] = (
    data["ctr"] < data["ctr"].median()
).astype(int)

data.head()

## select features

In [ ]:
feature_cols = [

"impressions",

"clicks",

"avg_position",

"visible_queries",

"position_volatility"

]

model_data = data.dropna(subset=feature_cols)

X = model_data[feature_cols]

y = model_data["low_ctr"]

groups = model_data["client_hash_id"]

## Group split

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

## Train the model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

## Evaluate

In [ ]:
from sklearn.metrics import classification_report

predictions = model.predict(X_test)

print(classification_report(
    y_test,
    predictions,
    digits=3
))

## Feature Importance

In [ ]:
importance = pd.DataFrame({

"Feature": feature_cols,

"Importance": model.feature_importances_

})

importance.sort_values(
    "Importance",
    ascending=False
)

## Ranked recommendation

In [ ]:
model_data["priority_score"] = model.predict_proba(X)[:,1]

recommendations = model_data.sort_values(
    "priority_score",
    ascending=False
)

recommendations[
[
"client_hash_id",
"content_hash_id",
"priority_score"
]
].head(20)

## ABstract

This project investigates whether historical search-performance signals can identify pages that should be prioritized for content refresh.

A machine-learning model was trained using anonymized FlyRank warehouse data.

The model predicts which pages are likely to have low click-through performance.

The resulting predictions are converted into a ranked recommendation list for content review.

The project is intended as decision-support rather than automated decision-making.

## Problem statement

Content teams manage thousands of pages, making manual review difficult.

This project builds a repeatable machine-learning workflow that ranks pages according to refresh priority.

The output helps content teams decide which pages should be reviewed first.

Incorrect recommendations may result in unnecessary review effort or missed optimization opportunities.

The model supports human decision-making and does not replace editorial judgment.

## Methodology

The project uses the FlyRank internship warehouse hosted on Hugging Face.

Features are aggregated from historical search-performance signals.

A Random Forest classifier is trained using GroupShuffleSplit so that pages from the same client do not appear in both the training and testing sets.

Feature importance is examined to understand which variables contribute most strongly to the predictions.

The final output is a ranked recommendation list ordered by predicted refresh priority.

## Limitation

The model identifies statistical associations rather than causal relationships.

The warehouse contains anonymized data and does not include business context.

Predictions should be interpreted as decision-support recommendations.

Future validation using later time windows would improve confidence in the model.

# Conclusion

This project demonstrates that historical search-performance signals can be used to rank pages according to refresh priority.

A machine-learning approach captures interactions between multiple signals that would be difficult to express using fixed rules alone.

The resulting ranking can help content teams focus review efforts on pages with a higher likelihood of decline.

The model should be used as a decision-support tool rather than a replacement for human judgment.